# Local Evaluation - Groundedness

After you have setup and configured the prompt flow, its time to evaluation its performance. Here we can use the prompt flow SDK to test different questions and see how the prompt flow performs using the evaluation prompt flows provided.

In [1]:
from promptflow import PFClient

pf_client = PFClient()

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Add a question to test the base prompt flow.
question = "Can you tell me about your jackets?"
customerId = "4"
output = pf_client.test(
    flow="../contoso-chat", # Path to the flow directory
    inputs={ # Inputs to the flow
        "chat_history": [],
        "question": question,
        "customerId": customerId,
    },
)

output["answer"] = "".join(list(output["answer"]))

[2024-02-27 18:52:31,441][promptflow._sdk.entities._connection][WARNING] - Please use connection.secrets[key] to access secrets.


2024-02-27 18:52:31 +0000    2580 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-02-27 18:52:31 +0000    2580 execution.flow     INFO     Start to run 5 nodes with concurrency level 16.
2024-02-27 18:52:31 +0000    2580 execution.flow     INFO     Executing node question_embedding. node run id: c887b52d-2348-4375-a65b-12042143f201_question_embedding_0
2024-02-27 18:52:31 +0000    2580 execution.flow     INFO     Executing node customer_lookup. node run id: c887b52d-2348-4375-a65b-12042143f201_customer_lookup_0
2024-02-27 18:52:31 +0000    2580 execution.flow     INFO     Node customer_lookup completes.
2024-02-27 18:52:32 +0000    2580 execution.flow     INFO     Node question_embedding completes.
2024-02-27 18:52:32 +0000    2580 execution.flow     INFO     Executing node retrieve_documentation. node run id: c887b52d-2348-4375-a65b-12042143f201_retrieve_documentation_0
2024-02-27 18:52:32 +0000    2580 execution.flow     INFO     Node retrieve_documentatio

In [3]:
output

{'answer': "Hey Sarah Lee! 🌟 We have two amazing jackets to keep you cozy on your outdoor adventures! 🏞️ \n\n1. The Summit Breeze Jacket (Product ID: 3) is a lightweight, windproof jacket with water-resistant fabric and reflective accents for night treks. It's a perfect companion for hiking. \n2. The RainGuard Hiking Jacket (Product ID: 17) offers waterproof, breathable fabric, adjustable hood, multiple pockets, and ventilation zippers. It's ideal for various outdoor activities. \n\nHope that helps! Let me know if you need anything else. 😊",
 'context': [{'id': '3',
   'title': 'Summit Breeze Jacket',
   'content': "Discover the joy of hiking with MountainStyle's Summit Breeze Jacket. This lightweight jacket is your perfect companion for outdoor adventures. Sporting a trail-ready, windproof design and a water-resistant fabric, it's ready to withstand any weather. The breathable polyester material and adjustable cuffs keep you comfortable, whether you're ascending a mountain or strollin

Test the groundedness of the prompt flow with the answer from the above question.

In [4]:
test = pf_client.test(
    flow="groundedness",
    inputs={
        "question": question,
        "context": str(output["context"]),
        "answer": output["answer"],
    },
)

2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Start to run 2 nodes with concurrency level 16.
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Executing node groundedness_score. node run id: 73ba7e3c-5b6b-4496-b3cb-853175a45df8_groundedness_score_0
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Node groundedness_score completes.
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Executing node concat_scores. node run id: 73ba7e3c-5b6b-4496-b3cb-853175a45df8_concat_scores_0
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Node concat_scores completes.
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Start to run 1 nodes with concurrency level 16.
2024-02-27 18:52:34 +0000    2580 execution.flow     INFO     Executing node aggregate_variants_results. node run id: fa7da54b-9562-486f-a853-1c0620aac946_aggre

In [5]:
test

{'gpt_groundedness': 5.0}

# Local Evaluation - Multiple Metrics 

Now use the same prompt flow and test it against the Multi Evaluation flow for groundedness, coherence, fluency, and relevance.

In [6]:
test_multi = pf_client.test(
    "multi_flow",
    inputs={
        "question": question,
        "context": str(output["context"]),
        "answer": output["answer"],
    },
)

2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Start executing nodes in thread pool mode.
2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Start to run 8 nodes with concurrency level 16.
2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Executing node coherence_score. node run id: 322eb917-30b8-40a5-907c-02d02d13a24c_coherence_score_0
2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Executing node fluency_score. node run id: 322eb917-30b8-40a5-907c-02d02d13a24c_fluency_score_0
2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Executing node groundedness_score. node run id: 322eb917-30b8-40a5-907c-02d02d13a24c_groundedness_score_0
2024-02-27 18:52:35 +0000    2580 execution.flow     INFO     Executing node relevance_score. node run id: 322eb917-30b8-40a5-907c-02d02d13a24c_relevance_score_0
2024-02-27 18:52:36 +0000    2580 execution.flow     INFO     Node relevance_score completes.
2024-02-27 18:52:36 +0000    2580 executi

In [7]:
test_multi

{'gpt_coherence': 5.0,
 'gpt_fluency': 5.0,
 'gpt_groundedness': 5.0,
 'gpt_relevance': 5.0}

# AI Studio Azure batch run on an evaluation json dataset

Now in order to test these more thoroughly, we can use the Azure AI Studio to run batches of test data with the evaluation prompt flow on a larger dataset.

In [8]:
import json
# Import required libraries
from promptflow.azure import PFClient

# Import required libraries
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential

In [9]:
try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

Populate the `config.json` file with the subscription_id, resource_group, and workspace_name.

In [10]:
config_path = "../config.json"
pf_azure_client = PFClient.from_config(credential=credential, path=config_path)

Found the config file in: ../config.json


Add the runtime from the AI Studio that will be used for the cloud batch runs.

In [11]:
# Update the runtime to the name of the runtime you created previously
runtime = "automatic"
# load flow
flow = "../contoso-chat"
# load data
data = "../data/salestestdata.jsonl"

In [12]:
# get current time stamp for run name
import datetime
now = datetime.datetime.now()
timestamp = now.strftime("%Y_%m_%d_%H%M%S")
run_name = timestamp+"chat_base_run"
print(run_name)

2024_02_27_185259chat_base_run


Create a base run to use as the variant for the evaluation runs. 

_NOTE: If you get "'An existing connection was forcibly closed by the remote host'" run the cell again._

In [13]:
# create base run in Azure Ai Studio
base_run = pf_azure_client.run(
    flow=flow,
    data=data,
    column_mapping={
        # reference data
        "customerId": "${data.customerId}",
        "question": "${data.question}",
    },
    runtime=runtime,
    # create a display name as current datetime
    display_name=run_name,
    name=run_name
)
print(base_run)

Uploading contoso-chat (0.68 MBs): 100%|██████████| 679466/679466 [00:00<00:00, 2864875.99it/s]


[2024-02-27 18:53:01,997][promptflow][WARNING] - You're using automatic runtime, if it's first time you're using it, it may take a while to build runtime and you may see 'NotStarted' status for a while. 


Portal url: https://ai.azure.com/projectflows/bulkrun/run/2024_02_27_185259chat_base_run/details?wsid=/subscriptions/fc0be99c-c8bc-44d9-b98a-7775c82fd517/resourcegroups/rg-contoso-chat-ai/providers/Microsoft.MachineLearningServices/workspaces/contoso-chat-ai-project
name: 2024_02_27_185259chat_base_run
created_on: '2024-02-27T18:53:07.176524+00:00'
status: NotStarted
display_name: 2024_02_27_185259chat_base_run
description:
tags: {}
properties:
  azureml.promptflow.runtime_name: automatic
  azureml.promptflow.runtime_version: 20240222.v3
  azureml.promptflow.definition_file_name: flow.dag.yaml
  azureml.promptflow.flow_lineage_id: a6117409b8b717c7e79942d8573769a6d1f4ac180e7fabad302590d2b43bff3c
  azureml.promptflow.flow_definition_datastore_name: workspaceblobstore
  azureml.promptflow.flow_definition_blob_path: LocalUpload/1bcff26c5fc4ff0ac24d633c25ba78c5/contoso-chat/flow.dag.yaml
  azureml.promptflow.inputs_mapping: '{"customerId":"${data.customerId}","question":"${data.question}"}'

In [14]:
pf_azure_client.stream(base_run)

(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'NotStarted', continue streaming...)
(Run status is 'Preparing', continue streaming...)
2024-02-27 18:57:40 +0000      51 promptflow-runtime INFO     [2024_02_27_185259chat_base_run] Receiving v2 bulk run request 02309144-439f-46ff-9261-80830cfb1263: {"flow_id": "2024_02_27_185259chat_base_run", "flow_run_id": "2024_02_27_185259chat_base_run", "flow_source": {"flow_source_type": 1, "flow_source_info": {"snapshot_id": "8306c5fe-0062-4479-907f-76aecfaa3697"}, "flow_dag_file": "flow.dag.yaml"}, "connections": "**data_scrubbed**", "log_path": "https://stcontosocha132474581317.blob.core.windows.net/da9630a2-6a9f-4b5e-b3b8-a2cc9fad5d4a-azureml/ExperimentRun/dcid.2024_02_

In [15]:
details = pf_azure_client.get_details(base_run)
details.head(10)

,inputs.customerId,inputs.question,inputs.line_number,inputs.chat_history,outputs.answer,outputs.context
outputs.line_number,,,,,,
0,4,tell me about your hiking jackets,0,[],Hey Sarah Lee! 🌲 Our hiking jackets are perfec...,"[{'id': '17', 'title': 'RainGuard Hiking Jacke..."
1,1,Do you have any climbing gear?,1,[],"Hey John! Absolutely, we have the SummitClimbe...","[{'id': '9', 'title': 'SummitClimber Backpack'..."
2,3,Can you tell me about your selection of tents?,2,[],Hey Michael! 🏕️ We have a great selection of t...,"[{'id': '15', 'title': 'SkyView 2-Person Tent'..."
3,6,Do you have any hiking boots?,3,[],"Hey Emily Rodriguez! Absolutely, we have the T...","[{'id': '4', 'title': 'TrekReady Hiking Boots'..."
4,2,What gear do you recommend for hiking?,4,[],"Hey Jane Doe! For hiking, I'd recommend pairin...","[{'id': '10', 'title': 'TrailBlaze Hiking Pant..."


# Cloud Eval run on Json Data

In [16]:
eval_flow = "multi_flow/"
data = "../data/salestestdata.jsonl"
run_name = timestamp+"chat_eval_run"
print(run_name)

eval_run_variant = pf_azure_client.run(
    flow=eval_flow,
    data=data,  # path to the data file
    run=base_run,  # use run as the variant
    column_mapping={
        # reference data
        "customerId": "${data.customerId}",
        "question": "${data.question}",
        "context": "${run.outputs.context}",
        # reference the run's output
        "answer": "${run.outputs.answer}",
    },
    runtime=runtime,
    display_name=run_name,
    name=run_name
)

2024_02_27_185259chat_eval_run


Uploading multi_flow (0.03 MBs): 100%|██████████| 31979/31979 [00:00<00:00, 117936.08it/s]


[2024-02-27 18:58:46,028][promptflow][WARNING] - You're using automatic runtime, if it's first time you're using it, it may take a while to build runtime and you may see 'NotStarted' status for a while. 


Portal url: https://ai.azure.com/projectflows/bulkrun/run/2024_02_27_185259chat_eval_run/details?wsid=/subscriptions/fc0be99c-c8bc-44d9-b98a-7775c82fd517/resourcegroups/rg-contoso-chat-ai/providers/Microsoft.MachineLearningServices/workspaces/contoso-chat-ai-project


In [17]:
pf_azure_client.stream(eval_run_variant)

(Run status is 'NotStarted', continue streaming...)
2024-02-27 18:59:40 +0000      51 promptflow-runtime INFO     [2024_02_27_185259chat_eval_run] Receiving v2 bulk run request e0f598d7-0a30-4eb0-ac12-4e01119dea1a: {"flow_id": "2024_02_27_185259chat_eval_run", "flow_run_id": "2024_02_27_185259chat_eval_run", "flow_source": {"flow_source_type": 1, "flow_source_info": {"snapshot_id": "ec5bfce6-b2b3-4271-a146-2e0ff8cb2514"}, "flow_dag_file": "flow.dag.yaml"}, "connections": "**data_scrubbed**", "log_path": "https://stcontosocha132474581317.blob.core.windows.net/da9630a2-6a9f-4b5e-b3b8-a2cc9fad5d4a-azureml/ExperimentRun/dcid.2024_02_27_185259chat_eval_run/logs/azureml/executionlogs.txt?sv=2019-07-07&sr=b&sig=**data_scrubbed**&skoid=304984a7-1fbe-4be9-9112-c09b50ec317f&sktid=d10c1b57-37dd-4393-92fe-44f4035f4d9a&skt=2024-02-27T18%3A43%3A05Z&ske=2024-02-29T02%3A53%3A05Z&sks=b&skv=2019-07-07&st=2024-02-27T18%3A49%3A39Z&se=2024-02-28T02%3A59%3A39Z&sp=rcw", "app_insights_instrumentation_key": "I

In [18]:
details = pf_azure_client.get_details(eval_run_variant)
details.head(10)

,inputs.customerId,inputs.question,inputs.context,inputs.answer,inputs.line_number,inputs.chat_history,outputs.gpt_coherence,outputs.gpt_fluency,outputs.gpt_groundedness,outputs.gpt_relevance
outputs.line_number,,,,,,,,,,
0,4,tell me about your hiking jackets,"[{'id': '17', 'title': 'RainGuard Hiking Jacke...",Hey Sarah Lee! 🌲 Our hiking jackets are perfec...,0,[],5.0,5.0,5.0,5.0
1,1,Do you have any climbing gear?,"[{'id': '9', 'title': 'SummitClimber Backpack'...","Hey John! Absolutely, we have the SummitClimbe...",1,[],4.0,5.0,3.0,5.0
2,3,Can you tell me about your selection of tents?,"[{'id': '15', 'title': 'SkyView 2-Person Tent'...",Hey Michael! 🏕️ We have a great selection of t...,2,[],4.0,4.0,5.0,3.0
3,6,Do you have any hiking boots?,"[{'id': '4', 'title': 'TrekReady Hiking Boots'...","Hey Emily Rodriguez! Absolutely, we have the T...",3,[],5.0,5.0,5.0,4.0
4,2,What gear do you recommend for hiking?,"[{'id': '10', 'title': 'TrailBlaze Hiking Pant...","Hey Jane Doe! For hiking, I'd recommend pairin...",4,[],4.0,5.0,5.0,5.0


In [19]:

metrics = pf_azure_client.get_metrics(eval_run_variant)
print(json.dumps(metrics, indent=4))

{
    "gpt_coherence": 4.4,
    "gpt_coherence_pass_rate(%)": 100.0,
    "gpt_fluency": 4.8,
    "gpt_fluency_pass_rate(%)": 100.0,
    "gpt_groundedness": 4.6,
    "gpt_groundedness_pass_rate(%)": 80.0,
    "gpt_relevance": 4.4,
    "gpt_relevance_pass_rate(%)": 80.0
}


In [20]:
pf_azure_client.visualize([base_run, eval_run_variant])

Web View: https://ml.azure.com/prompts/flow/bulkrun/runs/outputs?wsid=/subscriptions/fc0be99c-c8bc-44d9-b98a-7775c82fd517/resourceGroups/rg-contoso-chat-ai/providers/Microsoft.MachineLearningServices/workspaces/contoso-chat-ai-project&runId=2024_02_27_185259chat_base_run,2024_02_27_185259chat_eval_run
